Tic Tac Toe
---
Two players against each other

<img style="float:left" src="board.png" alt="drawing" width="200"/>

In [1]:
import numpy as np
import pickle

BOARD_ROWS = 3
BOARD_COLS = 3

In [24]:
class Board:
    def __init__(self):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS), dtype=int)
    
     # returns -1 is -1 wins, 1 if 1 wins and 0 for tie 
    def winner(self):
        # check rows
        for i in range(BOARD_ROWS):
            if np.abs(np.sum(self.board[i, :])) == 3:
                return np.sign(sum(self.board[i, :]))
        # check columns
        for i in range(BOARD_COLS):
            if np.abs(np.sum(self.board[:, i])) == 3: 
                return np.sign(sum(self.board[:,i]))
            
        # check diagonals
        diag_sum1 = sum([self.board[i, i] for i in range(BOARD_COLS)])
        diag_sum2 = sum([self.board[i, BOARD_COLS-i-1] for i in range(BOARD_COLS)])

        if (diag_sum1 == 3 or diag_sum2 == 3):
            return 1
        elif (diag_sum1 == -3 or diag_sum2 == -3):
            return -1
        
        # tie if no available positions
        if len(self.availablePositions()) == 0:
            return 0
        # not end
        return None # game not over yet
    
    # returns a list of available positions
    def availablePositions(self):
        positions = []
        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.board[i, j] == 0:
                    positions.append((i, j))  # need to be tuple
        return positions
    
    # updates the board state and switches to the next player
    def update(self, position, playerSymbol):
        self.board[position] = playerSymbol
    
    # board reset
    def reset(self):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
       # self.isEnd = False
       # self.playerSymbol = 1 # first player starts first with 1 

    # displays the board in a human-readable format
    def showBoard(self):
        # 1: x  -1: o
        for i in range(0, BOARD_ROWS):
            print('-------------')
            out = '| '
            for j in range(0, BOARD_COLS):
                if self.board[i, j] == 1:
                    token = 'x'
                if self.board[i, j] == -1:
                    token = 'o'
                if self.board[i, j] == 0:
                    token = ' '
                out += token + ' | '
            print(out)
        print('-------------')  

    # get a unique hash of current board state string representation of the board
    # board is a numpy array of shape (3, 3) representing the tic-tac-toe board
    def getHash(self): 
        boardHash = str(self.board.reshape(BOARD_COLS*BOARD_ROWS)) 
        # self.board.reshape(BOARD_COLS*BOARD_ROWS) flattens the 2D board into a 1D array, then converts to string
        # example: [[ 1  0 -1] [ 0  1  0] [-1  0  1]] -> '100010-101' 
        # how does this work? 1st row: 1 0 -1, 2nd row: 0 1 0, 3rd row: -1 0 1, 
        # then concatenates all elements into a single string = '[ 1  0 -1  0  1  0 -1  0  1]'
        # this becomes the unique hash for this board state.
        return boardHash  

In [8]:
# check BOARD class
new_board = Board()
new_board.updateState((0,0),-1)
new_board.updateState((1,1),-1)
new_board.updateState((2,2),-1)
new_board.showBoard()
print(new_board.winner())

-------------
| o |   |   | 
-------------
|   | o |   | 
-------------
|   |   | o | 
-------------
-1


### Game State
---
Reflect & Judge the state

2 players p1 and p2; p1 uses symbol1 = 1 and p2 uses symbol2 = -1, vacancy as 0

In [29]:
class Game:
    def __init__(self, p1, p2): # p1 = minmax or alphabeta player, p2 = human player
        self.board = Board() # np.zeros((BOARD_ROWS, BOARD_COLS), dtype=int

        self.p1 = p1 # first player - starts and has symbol 1
        self.p2 = p2 # second player follows first player and has symbol -1
        
        self.isEnd = False
    
    # returns -1 is p2 wins, 1 if p1 wins and 0 for tie and sets isEnd to True
    
    # plays the game
    # assumes Player class has a chooseAction(board, playerSymbol) that returns a tuple as 
    # the player action 
    def play(self):
        
        while not self.isEnd:

            # player 1 moves (starts the game) 
            p1_action = self.p1.chooseAction(self.board)
            
            # take action and upate board state
            self.board.update(p1_action, self.p1.playerSymbol) 

            # check board status if it is end
            self.board.showBoard()

            if self.check_win() is not None:
                self.isEnd = True
        
            # player2 moves
            p2_action = self.p2.chooseAction(self.board)
            self.board.update(p2_action, self.p2.playerSymbol)

            self.showBoard()

            if self.check_win() is not None:
                self.isEnd = True
    
    
    def check_win(self):
        win = self.board.winner()
        if win is not None: # if the game has ended, print the result and return  

            if win == 1:
                print(self.p1.name, "wins")
            elif win == -1:
                print(self.p2.name, "wins")
            else:
                print("Tie")
        return win
    
    def reset_game(self):
        self.board.reset()
        self.isEnd = False
        self.playerSymbol = 1


In [34]:
class HumanPlayer:
    def __init__(self, name, playerSymbol):
        self.name = name 
        self.playerSymbol = playerSymbol
    
    def chooseAction(self, current_board): 
        # need playerSymbol to determine which player is playing, 
        # but not used in this function, used in the class State 
        # to update the board state and switch to the next player
        positions = current_board.availablePositions()
        while True: # while a correct acion
            row = int(input("Input your action row:"))
            col = int(input("Input your action col:"))
            action = (row, col)
            if action in positions:
                return action

In [33]:
from copy import deepcopy # makes a deep copy of the board so that we can simulate moves without affecting the original board

class MinMaxPlayer:
    def __init__(self, name, playerSymbol):
        self.name = name
        self.playerSymbol = playerSymbol
        self.states_action = {} # store the best action you want to take in each state 
        # where the key is the hash of the state and the value is the value of that state

    # use this to store the value of a state in the states_values dictionary
    def store_value(self, board, best_action):
        hash_state = board.getHash()
        self.states_action[hash_state] = best_action

    def chooseAction(self, board):
        # make a copy of the board so that we can simulate moves without affecting the original board
        board_hash = board.getHash()
        if board.getHash() in self.states_action:
            return self.states_action[board.getHash()]

        current_board = deepcopy(board) # a new board a copy of the game board
        value,chosen_position = self.max_value(current_board) # chosen_position can be None, or a tuple
        
        #current_board[chosen_position[0]][chosen_position[1]] = playerSymbol # update the board with the chosen position
        self.store_value(current_board, chosen_position) # current_board is the board state after the move, value is the value of that state
        
        return chosen_position

    # max function   
    def max_value(self, board):
        val_current = board.winner()
        if val_current is not None: # val_current is a terminal state
            if val_current == self.playerSymbol:
                return 1, None
            elif val_current == -self.playerSymbol:
                return -1, None
            else:
                return 0, None
        
        # current_board is not a terminal state
        valid_positions = board.availablePositions()
        
        value = float("-inf")
        best_action = None

        new_board = deepcopy(board) # make a copy of the board so that we can simulate 
                                    # moves without affecting the original board
        for pos in valid_positions:
            new_board.update(pos,self.playerSymbol) # MAKE MOVE ON NEW BOARD
            
            # get new valid_positions
            min_val, _ = self.min_value(new_board)
            if min_val > value:
                value = min_val
                best_action = pos
            
            new_board.update(pos, 0) # UNDO THE MOVE
        
        return value, best_action
    
    # complete min_value method to return the minimum value and the chosen 
    # position for the min player
    def min_value(self, board):
        val_current = board.winner()
        if val_current is not None: # val_current is a terminal state
            if  val_current == self.playerSymbol:
                return 1, None
            elif val_current == -self.playerSymbol:
                return -1, None
            else:
                return 0, None
        
        # add code
        pass
        # return value, chosen_position

In [36]:
# test minmax
p1_minmax = MinMaxPlayer("Player1",1)
p2_minmax = MinMaxPlayer("Player2", -1)
game_minmax = Game(p1_minmax, p2_minmax)

In [37]:
class AlphaBeta:
    def __init__(self, name, playerSymbol):
        self.name = name
        self.playerSymbol = playerSymbol

    def chooseAction(self, board):
        return best_action

In [ ]:
import math
import random 

class MCTSNode:
    # player_to_move = symbol of the player who's turn is to make a move
    def __init__(self, board, player_to_move, move = None, parent = None):
        self.board = board
        self.player_to_move = player_to_move  # symbol to move at this node
        self.move = move                      # move that led here from parent
        self.parent = parent
        self.children = []
        self.untried_moves = board.available_positions()
        
        self.wins = 0
        self.visits = 0

    def is_terminal(self):
        return self.board.winner() is not None
    
    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, exploration_const = .7):
        # implement UCB: exploitation + exploration + picks non-explored children
        '''weights = []
         for child in self.children:
            if child.visits == 0:
               w = float("inf")# forces to chose an unexplored child 
            else
	            w = child.value/child.visits + exploration_const * math.sqrt(math.log(state.visits)/child.visits)
	         weights.append(w)
         distribution =[w / sum(weights) for w in weights]
         best_child = child with highest value in distribution
         return best_child # Node object
   '''

    # next_player = symbol for the next player (-1)*self.player_to_move
    def add_child(self, move, board_after_move, next_player):
        child = MCTSNode(
            board = board_after_move,
            player_to_move = next_player,
            move = move,
            parent = self
        )
        self.untried_moves.remove(move)
        self.children.append(child)
        return child

    def update(self, result):
        # result is from the root player's perspective:
        # +1 = root win, 0 = draw, -1 = root loss
        self.visits += 1
        self.wins   += result


class MCTSPlayer:
     # max_iter = maximum number of simulations for MCTS
   def __init__(self, name, playerSymbol, max_iter = 1000, exploration_const = .7):
      self.name = name
      self.max_iter = max_iter
      self.playerSymbol = playerSymbol
      self.exploration_const = exploration_const
      self.root = None

   def chooseAction(self, board):
      root = MCTSNode(deepcopy(board), self.playerSymbol)

      for i in range(self.iterations):
         self.MCTS_sample(self.root)

      if not self.root.children: # if no children
         return random.choice(board.available_positions())

      # return the child with most visit - most confidence in its value
      best_child = max(self.root.children, key = lambda child: child.visits)
      return best_child.move


   def MCTS_sample(self, node):
      node.visits += 1

      # 1) Terminal node
      if node.is_terminal():
         result = self.result_from_root_perspective(node.board.winner())
         node.wins += result
         return result

      # 2) Not fully expanded: expand one new child and rollout from there
      if not node.is_fully_expanded():
         child = node.expand()
         result = self.random_playout(child.board, child.player_to_move)
         child.update(result)
         node.wins += result
         return result

      # 3) Fully expanded: recurse into best UCB child
      child = node.best_child(self.exploration_weight)
      result = self.MCTS_sample(child)

      # 4) Backpropagate
      node.update(result)
      return result


   def rollout(self, board, player_to_move, root_symbol):
      sim_board = deepcopy(board)
      current_player = player_to_move

      while sim_board.winner() is None:
         moves = sim_board.available_positions()
         if not moves:
            break

         move = random.choice(moves)
         sim_board.update(move, current_player)
         current_player = -current_player

      winner = sim_board.winner()
      return self.result_from_root_perspective(winner)


   def result_from_root_perspective(self, winner):
      
      if winner == self.playerSymbol:
         return 1
      elif winner == -self.playerSsymbol:
         return -1
      return 0
    

   
       

### Human vs Computer

In [ ]:
p1 = MinMaxPlayer("computer", 1)
ph = HumanPlayer("human", -1)

st = Game(p1,ph)
st.play()